## Question 1

Let $\mathbf{A}$ be an array of activations of shape `float32[S_X, D_Y]` with `X * Y = N`. Do the following:

1. Write a function in JAX that computes the average within each `(X, Y)` shard, i.e. it returns an array of size `[X, Y]` where `arr[i, j]` is the average over shard `(i, j)`. Do this with both `jax.jit` and `shard_map`. Profile each and see how long they took. Was there any communication added? *Hint: there shouldn't be, but sometimes XLA adds it anyway.*

In [95]:
import jax
import jax.numpy as jnp
import jax.sharding as shd
import numpy as np
from jax.sharding import PartitionSpec as P

In [3]:
jax.config.update("jax_default_matmul_precision", "highest")

In [4]:
mesh = jax.make_mesh(axis_shapes=(4, 2), axis_names=("x", "y"))
shd.set_mesh(mesh)

In [5]:
def jit_avg(x: jnp.ndarray) -> jnp.ndarray:
    X, Y = mesh.shape["x"], mesh.shape["y"]
    i, j = x.shape
    x = x.reshape(X, i // X, Y, j // Y)
    return x.mean(axis=(1, 3))


@jax.shard_map(in_specs=P("x", "y"), out_specs=P("x", "y"))
def shard_map_avg(x: jnp.ndarray) -> jnp.ndarray:
    return x.mean(keepdims=True)

In [6]:
jitted_jit_avg = jax.jit(jit_avg)
jitted_shard_map_avg = jax.jit(shard_map_avg)

In [7]:
A, B = 16384, 16384
x = jnp.arange(A * B).reshape(A, B)
x = jax.device_put(x, P("x", "y"))

with jax.profiler.trace("/tmp/tensorboard"):
    y1 = jitted_jit_avg(x)
    y1.block_until_ready()

with jax.profiler.trace("/tmp/tensorboard"):
    y2 = jitted_shard_map_avg(x)
    y2.block_until_ready()

print(f"is same: {jnp.allclose(y1, y2)}")

2026-07-16 12:00:08.784580: E external/xla/xla/python/profiler/internal/python_hooks.cc:412] Can't import tensorflow.python.profiler.trace
2026-07-16 12:00:09.047319: E external/xla/xla/python/profiler/internal/python_hooks.cc:412] Can't import tensorflow.python.profiler.trace
2026-07-16 12:00:09.933630: E external/xla/xla/python/profiler/internal/python_hooks.cc:412] Can't import tensorflow.python.profiler.trace


is same: True


2026-07-16 12:00:10.152547: E external/xla/xla/python/profiler/internal/python_hooks.cc:412] Can't import tensorflow.python.profiler.trace


In [8]:
%load_ext tensorboard

In [9]:
%tensorboard --logdir /tmp/tensorboard

- jit version: 92.999us
- shard_map version: 92.989us

basically the same operation, no extra communication added

2. Write a function in JAX that returns `roll(x, shift, axis=0) - x` for some shift **within each shard along X**. I'm not enough of a masochist to make you do this in `jax.jit`, so just do this with `shard_map`.

In [10]:
@jax.shard_map(in_specs=(P("x", "y"), P()), out_specs=P("x", "y"))
def shard_map_roll(x: jnp.ndarray, shift: int) -> jnp.ndarray:
    return jnp.roll(x, shift, axis=0) - x


jitted_shard_map_roll = jax.jit(shard_map_roll)

In [11]:
A, B = 8, 8
x = jnp.arange(A * B).reshape(A, B)
x = jax.device_put(x, P("x", "y"))

"""
first shard of `x`:
[[0, 1, 2, 3],
 [8, 9, 10, 11]]

after roll:
[[8, 9, 10, 11],
 [0, 1, 2, 3]]

after roll - x:
[[8, 8, 8, 8],
 [-8, -8, -8, -8]]
"""

y1 = jitted_shard_map_roll(x, 1)
y1.block_until_ready()

print(y1)

[[ 8  8  8  8  8  8  8  8]
 [-8 -8 -8 -8 -8 -8 -8 -8]
 [ 8  8  8  8  8  8  8  8]
 [-8 -8 -8 -8 -8 -8 -8 -8]
 [ 8  8  8  8  8  8  8  8]
 [-8 -8 -8 -8 -8 -8 -8 -8]
 [ 8  8  8  8  8  8  8  8]
 [-8 -8 -8 -8 -8 -8 -8 -8]]


## Question 2

Here we'll make a basic "mixture of experts" model together. Let $\mathbf{W}$: `float32[E_X, D, F]` be a set of `E` "expert" matrices. Let $\mathbf{A}$: `float32[S_X, D]` (our activations) and let $\mathbf{B}$: `int32[S_X]` be a set of "routing assignments" where `B[i]` is an integer in the range `[0, E)` telling us which matrix we want to process that activation. We want to write a function in JAX that returns `Out[i] = A[i] @ W[B[i]]`.

1. Let's start by ignoring sharding altogether. Make all of these tensors small enough so they fit in one device. Write a local implementation of this function. *Make sure you don't materialize an array of shape `[S, D, F]`! Hint: try sorting the tokens into a new buffer of shape `[E, S, D]` with some attention to masking (why do we need the second dimension to have size `S`?).*

In [12]:
mesh = jax.make_mesh(axis_shapes=(8,), axis_names=("x",))
shd.set_mesh(mesh)

Mesh(device_ids=array([[0, 1],
       [2, 3],
       [7, 6],
       [5, 4]]), axis_names=('x', 'y'), axis_types=(Auto, Auto))

In [13]:
def moe_naive(W: jnp.ndarray, A: jnp.ndarray, B: jnp.ndarray) -> jnp.ndarray:
    out = jnp.einsum("sd, sdf -> sf", A, W[B], precision=jax.lax.Precision.HIGHEST)
    return out


"""
E = 2, S = 4

A: [S, D] = [tok0, tok1, tok2, tok3]
B: [S] = [0, 1, 1, 0]

order = jnp.argsort(B) = [0, 3, 1, 2]
B_sorted = B[order] = [0, 0, 1, 1]

rank = [0, 1, 0, 1] = [0, 1, 2, 3] - [0, 0, 2, 2]
rank = jnp.arange(S) - jnp.searchsorted(B_sorted, B_sorted, side="left")

buffer = jnp.zeros((E, S, D)).at[B_sorted, rank].set(A[order])
buffer: [E, S, D] = [
    [tok0, tok3, 0, 0],
    [tok1, tok2, 0, 0],
]

out = [E, S, F] = einsum("esd, edf -> esf", buffer, W)
gathered: [S, F] = out[B, rank]

rank: index inside each expert

return gathered
"""


def local_moe(W: jnp.ndarray, A: jnp.ndarray, B: jnp.ndarray) -> jnp.ndarray:
    E, D, F = W.shape
    S = A.shape[0]

    order = jnp.argsort(B)
    B_sorted = B[order]

    index = jnp.arange(S)
    is_start = jnp.concatenate([jnp.ones(1, dtype=bool), B_sorted[:-1] != B_sorted[1:]])
    start_index = jax.lax.cummax(jnp.where(is_start, index, 0))
    rank = index - start_index

    buffer = jnp.zeros((E, S, D), dtype=A.dtype).at[B_sorted, rank].set(A[order])
    out = jnp.einsum("esd, edf -> esf", buffer, W, precision=jax.lax.Precision.HIGHEST)
    result = jnp.zeros((S, F), dtype=A.dtype).at[order].set(out[B_sorted, rank])
    return result

2. If you just `jax.jit` the above method, something will happen. Profile this and see what communication it decided to do. How long does it take?

In [118]:
jitted_moe_naive = jax.jit(moe_naive)
jitted_local_moe = jax.jit(local_moe)

key = jax.random.PRNGKey(0)
A = jax.random.normal(key, (1024, 2048)).astype(jnp.bfloat16)
B = jax.random.randint(key, (1024,), 0, 8, dtype=jnp.int32)
W = jax.random.normal(key, (8, 2048, 8192)).astype(jnp.bfloat16)

A = jax.device_put(A, P("x", None))
B = jax.device_put(B, P("x"))
W = jax.device_put(W, P("x", None, None))

with jax.profiler.trace("/tmp/tensorboard"):
    y0 = jitted_moe_naive(W, A, B)
    y0.block_until_ready()

with jax.profiler.trace("/tmp/tensorboard"):
    y1 = jitted_local_moe(W, A, B)
    y1.block_until_ready()

np.testing.assert_allclose(y0, y1, rtol=1e-2, atol=1e-2)

2026-07-16 12:54:16.651118: E external/xla/xla/python/profiler/internal/python_hooks.cc:412] Can't import tensorflow.python.profiler.trace
2026-07-16 12:54:20.772766: E external/xla/xla/python/profiler/internal/python_hooks.cc:412] Can't import tensorflow.python.profiler.trace
2026-07-16 12:54:21.739783: E external/xla/xla/python/profiler/internal/python_hooks.cc:412] Can't import tensorflow.python.profiler.trace
2026-07-16 12:54:26.938197: E external/xla/xla/python/profiler/internal/python_hooks.cc:412] Can't import tensorflow.python.profiler.trace


In [ ]:
%tensorboard --logdir /tmp/tensorboard

Reusing TensorBoard on port 6006 (pid 13832), started 0:00:22 ago. (Use '!kill 13832' to kill it.)

Takes about 1ms, mostly communication. MatMul only takes 44.700us

total FLOPs = 2 * E * S * D * F
est time (MatMul) = 2 * E * S * D * F / (8 * tpuv6e FLOPs/s) = 37us -> pretty similar

1. AllGather_X (A[S_X, D]) = A[S, D] (44.784us)

est time = 2 * S * D / 9e10 = 46us

in order to fill in the buffer with A

2. AllReduce(A) = all_reduce.1 (45.015us)

est time = 2 * 2 * S * D / 9e10 = 46 * 2us

for some reason allreduce is having 2x bandwidth (??)

--> actual time is just 46us

-> to collect A using the indexes but XLA lowers it down to all reduce

3. AllGather_X (B[S_X]) = B[S] (9.821us)

est time = 2 * S / 9e10 = 22ns (X)
latency = 1us * 8 (hops) = 1us

-> latency bound

in order to argsort B

4. AllReduce(Result) = all_reduce.2 (168.306us)

est time = 2 * 2 * S * F / 9e10 = 372us

for some reason allreduce is having 2x bandwidth (??)

--> actual time is just 186us

-> To create the result array using indicies


3. One problem you'll notice with the above is that it likely gathers the full set of activations `A` locally, i.e. $\text{AllGather}_X([S_X, D])$. Not only is this expensive communication-wise, it's also incredibly expensive memory-wise if we can't fit the full set of activations locally. Implement the above using `shard_map` and explicit communication.

    1. For a first pass, it might be easiest to use a `jax.lax.all_gather` and reorder as in step 1.

    2. For a second pass, try to avoid materializing any array of size `[E, S, D]`, i.e. try to perform the computation in a ragged fashion using a `jax.lax.all_to_all` inside a `jax.lax.while_loop`. This way, you can avoid materializing the full activations and wasting compute on padding. How much faster is this than your original implementation?

In [17]:
"""
idea: use `all_to_all` and `while_loop` to avoid materializing the full buffer

current implementation:
> init full buffer
> fill buffer with A
> matmul with W

new implementation:
> Sort inside each shard B[S_X] -> B_sorted[S_X]
> Make small buffer buf[E, S // X, D]
> Fill buffer with A[S_X, D]
> 

- all_to_all requires transfer of same amount of data from each device
- while loop stops when every device has only padding
- need to know beforehand how long the loop will run by exchanging metadata

> count the number of tokens per expert in each shard
> get the maximum
> all gather the maximum
> run the loop for this maximum
"""


def shard_map_moe(W: jnp.ndarray, A: jnp.ndarray, B: jnp.ndarray) -> jnp.ndarray:
    x = jax.lax.axis_size("x")
    Ex, D, F = W.shape
    E = Ex * x
    Sx = A.shape[0]

    max_expert_shard = jnp.max(jnp.bincount(B, length=E))
    max_expert_gathered = jax.lax.all_gather(max_expert_shard[None], axis_name="x", tiled=True)
    max_expert = jnp.max(max_expert_gathered)

    order = jnp.argsort(B)
    B_sorted = B[order]
    index = jnp.arange(Sx)
    is_start = jnp.concatenate([jnp.ones(1, dtype=bool), B_sorted[:-1] != B_sorted[1:]])
    start_index = jax.lax.cummax(jnp.where(is_start, index, 0))
    rank = index - start_index

    buf = jnp.zeros((E, Sx, D), dtype=A.dtype)
    buf = buf.at[B_sorted, rank].set(A[order])

    chunk_size = 4

    def body_fn(carry):
        accum, start = carry

        chunk = jax.lax.dynamic_slice_in_dim(buf, start, chunk_size, axis=1)
        chunk = jax.lax.all_to_all(chunk, axis_name="x", split_axis=0, concat_axis=1, tiled=True)
        update = jnp.einsum("ecd, edf -> ecf", chunk, W)
        update = jax.lax.all_to_all(update, axis_name="x", split_axis=1, concat_axis=0, tiled=True)

        accum = jax.lax.dynamic_update_slice_in_dim(accum, update, start, axis=1)
        return accum, start + chunk_size

    accum = jnp.zeros((E, Sx, F), dtype=A.dtype)
    accum = jax.lax.pvary(accum, ("x",))
    accum, _ = jax.lax.while_loop(lambda carry: carry[1] < max_expert, body_fn, (accum, 0))

    out = jnp.zeros((Sx, F), dtype=A.dtype).at[order].set(accum[B_sorted, rank])
    return out

In [119]:
jitted_shard_map_moe = jax.jit(
    jax.shard_map(
        shard_map_moe,
        in_specs=(P("x", None, None), P("x", None), P("x")),
        out_specs=P("x", None),
    )
)

key = jax.random.PRNGKey(0)
A = jax.random.normal(key, (1024, 2048)).astype(jnp.bfloat16)
B = jax.random.randint(key, (1024,), 0, 8, dtype=jnp.int32)
W = jax.random.normal(key, (8, 2048, 8192)).astype(jnp.bfloat16)

A = jax.device_put(A, P("x", None))
B = jax.device_put(B, P("x"))
W = jax.device_put(W, P("x", None, None))

with jax.profiler.trace("/tmp/tensorboard"):
    y0 = jitted_local_moe(W, A, B)
    y0.block_until_ready()

with jax.profiler.trace("/tmp/tensorboard"):
    y_shardmap = jitted_shard_map_moe(W, A, B)
    y_shardmap.block_until_ready()

np.testing.assert_allclose(y0, y_shardmap, rtol=1e-2, atol=1e-2)

2026-07-16 12:55:09.152310: E external/xla/xla/python/profiler/internal/python_hooks.cc:412] Can't import tensorflow.python.profiler.trace
2026-07-16 12:55:09.253067: E external/xla/xla/python/profiler/internal/python_hooks.cc:412] Can't import tensorflow.python.profiler.trace
2026-07-16 12:55:10.163282: E external/xla/xla/python/profiler/internal/python_hooks.cc:412] Can't import tensorflow.python.profiler.trace
2026-07-16 12:55:13.437081: E external/xla/xla/python/profiler/internal/python_hooks.cc:412] Can't import tensorflow.python.profiler.trace


In [ ]:
%tensorboard --logdir /tmp/tensorboard

Reusing TensorBoard on port 6006 (pid 9909), started 0:00:11 ago. (Use '!kill 9909' to kill it.)

2x faster = 500us

4. Most MoEs route to multiple (`k`) experts and then average the result. Refactor the above to implement this. Let $\mathbf{B}$: `int32[S_X, k]` in this case for the `k` experts to route to.

In [19]:
"""
B: [S, k]
0 1
3 1
2 4
2 3

flatten: 0 1 3 1 2 4 2 3
argsort: 0 1 3 4 6 2 7 5
sorted : 0 1 1 2 2 3 3 4
rank   : 0 0 1 0 1 0 1 0

sorted = unsorted[argsort]
unsorted[argsort] = sorted

buf:
[tok0, pad, pad, pad]
[tok1, tok3, pad, pad]
[tok4, tok6, pad, pad]
[tok2, tok7, pad, pad]
[tok5, pad, pad, pad]


> flatten B -> B_flat[S * k]
> argsort B_flat -> B_sorted_flat[S * k]
> do the same calculation as before

> gathering should be different
> gathered shape: [k*S, F]
> reshape to [k, S, F]
> average over k
"""


"""
idea: use `all_to_all` and `while_loop` to avoid materializing the full buffer

current implementation:
> init full buffer
> fill buffer with A
> matmul with W

new implementation:
> Sort inside each shard B[S_X] -> B_sorted[S_X]
> Make small buffer buf[E, S // X, D]
> Fill buffer with A[S_X, D]
> 

- all_to_all requires transfer of same amount of data from each device
- while loop stops when every device has only padding
- need to know beforehand how long the loop will run by exchanging metadata

> count the number of tokens per expert in each shard
> get the maximum
> all gather the maximum
> run the loop for this maximum
"""


def shard_map_moe_k(W: jnp.ndarray, A: jnp.ndarray, B: jnp.ndarray) -> jnp.ndarray:
    Ex, D, F = W.shape
    E = Ex * jax.lax.axis_size("x")
    Sx, k = B.shape

    B_flat = B.reshape(-1)
    max_expert_shard = jnp.max(jnp.bincount(B_flat, length=E))
    max_expert_gathered = jax.lax.all_gather(max_expert_shard[None], axis_name="x", tiled=True)
    max_expert = jnp.max(max_expert_gathered)

    order = jnp.argsort(B_flat)
    B_sorted = B_flat[order]
    index = jnp.arange(Sx * k)
    is_start = jnp.concatenate([jnp.ones(1, dtype=bool), B_sorted[:-1] != B_sorted[1:]])
    start_index = jax.lax.cummax(jnp.where(is_start, index, 0))
    rank = index - start_index

    buf = jnp.zeros((E, Sx * k, D), dtype=A.dtype)
    buf = buf.at[B_sorted, rank].set(A[order // k])

    chunk_size = 4

    def body_fn(carry):
        accum, start = carry

        chunk = jax.lax.dynamic_slice_in_dim(buf, start, chunk_size, axis=1)
        chunk = jax.lax.all_to_all(chunk, axis_name="x", split_axis=0, concat_axis=1, tiled=True)
        update = jnp.einsum("ecd, edf -> ecf", chunk, W)
        update = jax.lax.all_to_all(update, axis_name="x", split_axis=1, concat_axis=0, tiled=True)

        accum = jax.lax.dynamic_update_slice_in_dim(accum, update, start, axis=1)
        return accum, start + chunk_size

    accum = jnp.zeros((E, Sx * k, F), dtype=A.dtype)
    accum = jax.lax.pvary(accum, ("x",))
    accum, _ = jax.lax.while_loop(lambda carry: carry[1] < max_expert, body_fn, (accum, 0))

    out = jnp.zeros((Sx * k, F), dtype=A.dtype).at[order].set(accum[B_sorted, rank])
    out = out.reshape(Sx, k, F).mean(axis=1)
    return out

In [25]:
jitted_shard_map_moe_k = jax.jit(
    jax.shard_map(
        shard_map_moe_k,
        in_specs=(P("x", None, None), P("x", None), P("x", None)),
        out_specs=P("x", None),
    )
)

key = jax.random.PRNGKey(0)
A = jax.random.normal(key, (1024, 2048)).astype(jnp.bfloat16)
B = jax.random.randint(key, (1024, 2), 0, 8, dtype=jnp.int32)
W = jax.random.normal(key, (8, 2048, 8192)).astype(jnp.bfloat16)

A = jax.device_put(A, P("x", None))
B = jax.device_put(B, P("x", None))
W = jax.device_put(W, P("x", None, None))

with jax.profiler.trace("/tmp/tensorboard"):
    y0 = jitted_local_moe(W, A, B[:, 0])
    y0.block_until_ready()

    y1 = jitted_local_moe(W, A, B[:, 1])
    y1.block_until_ready()

with jax.profiler.trace("/tmp/tensorboard"):
    y_shardmap = jitted_shard_map_moe_k(W, A, B)
    y_shardmap.block_until_ready()

np.testing.assert_allclose((y0 + y1) / 2, y_shardmap, rtol=1e-2, atol=1e-2)

2026-07-16 12:01:47.518473: E external/xla/xla/python/profiler/internal/python_hooks.cc:412] Can't import tensorflow.python.profiler.trace
2026-07-16 12:01:49.503291: E external/xla/xla/python/profiler/internal/python_hooks.cc:412] Can't import tensorflow.python.profiler.trace
2026-07-16 12:01:50.538545: E external/xla/xla/python/profiler/internal/python_hooks.cc:412] Can't import tensorflow.python.profiler.trace
2026-07-16 12:01:53.495004: E external/xla/xla/python/profiler/internal/python_hooks.cc:412] Can't import tensorflow.python.profiler.trace


## Question 3

The collective matmul example above is actually super relevant for real LLMs. Let's tweak the example to do the full Transformer stack.

1. As an exercise, let's start by implementing an AllReduce collective matmul, i.e. `A[B_X, D_Y]` $*_D$ `W[D_Y, F]` $\rightarrow$ `Out[B_X, F]`. Note that the output isn't replicated. The naive algorithm is discussed above, basically just a local matmul followed by an AllReduce. Try to make a comms overlapped "collective" version of this operation. *Hint: tile over the output dimension and feel free to use `jax.lax.psum` (aka AllReduce). Note: due to the way XLA handles this, it may not actually be faster than the baseline.*

In [26]:
mesh = jax.make_mesh(axis_shapes=(2, 4), axis_names=("x", "y"))
shd.set_mesh(mesh)

Mesh(device_ids=array([0, 1, 2, 3, 7, 6, 5, 4]), axis_names=('x',), axis_types=(Auto,))

In [27]:
"""
split the W to [D_Y, F // X] chunks and rotate the X dim of W along the X shard while doing all_to_all in Y axis
"""


def naive_matmul_allreduce(lhs: jnp.ndarray, rhs: jnp.ndarray) -> jnp.ndarray:
    out = lhs @ rhs
    out = jax.lax.with_sharding_constraint(out, P("x", None))
    return out


def collective_matmul_allreduce(lhs: jnp.ndarray, rhs: jnp.ndarray) -> jnp.ndarray:
    _, F = rhs.shape

    chunk_size = 4096
    assert F % chunk_size == 0

    def body_fn(_: None, i: int):
        rhs_chunk = jax.lax.dynamic_slice_in_dim(rhs, i * chunk_size, chunk_size, axis=1)
        update = lhs @ rhs_chunk
        update = jax.lax.psum(update, axis_name="y")
        return None, update

    _, out = jax.lax.scan(body_fn, None, jnp.arange(F // chunk_size), unroll=True)
    out = jnp.concatenate(out, axis=1)

    return out

In [28]:
jitted_naive_matmul_allreduce = jax.jit(naive_matmul_allreduce)
jitted_collective_matmul_allreduce = jax.jit(
    jax.shard_map(
        collective_matmul_allreduce,
        in_specs=(P("x", "y"), P("y", None)),
        out_specs=P("x", None),
    )
)

key = jax.random.PRNGKey(0)
A = jax.random.randint(key, (16384, 2048), 0, 16).astype(jnp.bfloat16)
B = jax.random.randint(key, (2048, 8192), 0, 16).astype(jnp.bfloat16)

A = jax.device_put(A, P("x", "y"))
B = jax.device_put(B, P("y", None))

with jax.profiler.trace("/tmp/tensorboard"):
    y = jitted_naive_matmul_allreduce(A, B)
    y.block_until_ready()

with jax.profiler.trace("/tmp/tensorboard"):
    y1 = jitted_collective_matmul_allreduce(A, B)
    y1.block_until_ready()

np.testing.assert_allclose(y, y1, rtol=1e-2, atol=1e-2)

2026-07-16 12:02:10.809283: E external/xla/xla/python/profiler/internal/python_hooks.cc:412] Can't import tensorflow.python.profiler.trace
2026-07-16 12:02:11.391823: E external/xla/xla/python/profiler/internal/python_hooks.cc:412] Can't import tensorflow.python.profiler.trace
2026-07-16 12:02:12.286431: E external/xla/xla/python/profiler/internal/python_hooks.cc:412] Can't import tensorflow.python.profiler.trace
2026-07-16 12:02:13.463434: E external/xla/xla/python/profiler/internal/python_hooks.cc:412] Can't import tensorflow.python.profiler.trace


In [29]:
%tensorboard --logdir /tmp/tensorboard

Reusing TensorBoard on port 6006 (pid 9909), started 0:02:03 ago. (Use '!kill 9909' to kill it.)

The collective version is actually slower, since we cannot effectively overlap AllReduce with the computation.

- Raw: 1.296ms
- Chunked (chunk num = 2): 1.706ms

Performance gets worse as we split more chunks, since it is not really effective doing small matmuls and transferring them around

2. The complement to the AllReduce collective matmul above is a ReduceScatter collective matmul, as in `Tmp[B_X, F_Y]` $*_F$ `W2[F_Y, D]` $\rightarrow$ `Out[B_X, D_Y]`. This occurs in the down-projection matrix in a Transformer. Implement a collective, overlapped version of this in JAX. Be careful about passing only the minimal amount of data you need. *Hint: try permuting the result as you accumulate it.*

In [53]:
def naive_matmul_reducescatter(lhs: jnp.ndarray, rhs: jnp.ndarray) -> jnp.ndarray:
    out = lhs @ rhs
    out = jax.lax.with_sharding_constraint(out, P("x", "y"))
    return out


def collective_matmul_reducescatter_scan(lhs: jnp.ndarray, rhs: jnp.ndarray) -> jnp.ndarray:
    _, D = rhs.shape

    axis_size = jax.lax.axis_size("y")
    idx = jax.lax.axis_index("y")

    chunk_size = D // axis_size

    perm = [(j, (j - 1) % axis_size) for j in range(axis_size)]

    def body_fn(accum: jnp.ndarray, i: int):
        accum = jax.lax.ppermute(accum, axis_name="y", perm=perm)
        rhs_chunk = jax.lax.dynamic_slice_in_dim(
            rhs, (idx + i + 1) % axis_size * chunk_size, chunk_size, axis=1
        )
        update = lhs @ rhs_chunk
        return accum + update, update

    rhs_chunk = jax.lax.dynamic_slice_in_dim(
        rhs, (idx + 1) % axis_size * chunk_size, chunk_size, axis=1
    )
    accum = lhs @ rhs_chunk
    out, _ = jax.lax.scan(body_fn, accum, jnp.arange(1, axis_size), unroll=True)

    return out.astype(lhs.dtype)


def collective_matmul_reducescatter_loop(lhs: jnp.ndarray, rhs: jnp.ndarray) -> jnp.ndarray:
    _, D = rhs.shape

    axis_size = jax.lax.axis_size("y")
    idx = jax.lax.axis_index("y")

    chunk_size = D // axis_size

    perm = [(j, (j - 1) % axis_size) for j in range(axis_size)]

    def body_fn(accum: jnp.ndarray, i: int):
        accum = jax.lax.ppermute(accum, axis_name="y", perm=perm)
        rhs_chunk = jax.lax.dynamic_slice_in_dim(
            rhs, (idx + i + 1) % axis_size * chunk_size, chunk_size, axis=1
        )
        accum += lhs @ rhs_chunk
        return accum

    rhs_chunk = jax.lax.dynamic_slice_in_dim(
        rhs, (idx + 1) % axis_size * chunk_size, chunk_size, axis=1
    )
    accum = lhs @ rhs_chunk
    for i in range(1, axis_size):
        accum = body_fn(accum, i)

    return accum

In [54]:
jitted_naive_matmul_reducescatter = jax.jit(naive_matmul_reducescatter)
jitted_collective_matmul_reducescatter_scan = jax.jit(
    jax.shard_map(
        collective_matmul_reducescatter_scan,
        in_specs=(P("x", "y"), P("y", None)),
        out_specs=P("x", "y"),
    )
)
jitted_collective_matmul_reducescatter_loop = jax.jit(
    jax.shard_map(
        collective_matmul_reducescatter_loop,
        in_specs=(P("x", "y"), P("y", None)),
        out_specs=P("x", "y"),
    )
)

key = jax.random.PRNGKey(0)
A = jax.random.randint(key, (16384, 8192), 0, 16).astype(jnp.bfloat16)
B = jax.random.randint(key, (8192, 2048), 0, 16).astype(jnp.bfloat16)

A = jax.device_put(A, P("x", "y"))
B = jax.device_put(B, P("y", None))

with jax.profiler.trace("/tmp/tensorboard"):
    y = jitted_naive_matmul_reducescatter(A, B)
    y.block_until_ready()

with jax.profiler.trace("/tmp/tensorboard"):
    y1 = jitted_collective_matmul_reducescatter_scan(A, B)
    y1.block_until_ready()

with jax.profiler.trace("/tmp/tensorboard"):
    y2 = jitted_collective_matmul_reducescatter_loop(A, B)
    y2.block_until_ready()

np.testing.assert_allclose(y, y1, rtol=1e-2, atol=1e-2)
np.testing.assert_allclose(y, y2, rtol=1e-2, atol=1e-2)

2026-07-16 12:15:05.990708: E external/xla/xla/python/profiler/internal/python_hooks.cc:412] Can't import tensorflow.python.profiler.trace
2026-07-16 12:15:07.395777: E external/xla/xla/python/profiler/internal/python_hooks.cc:412] Can't import tensorflow.python.profiler.trace
2026-07-16 12:15:08.308187: E external/xla/xla/python/profiler/internal/python_hooks.cc:412] Can't import tensorflow.python.profiler.trace
2026-07-16 12:15:10.230620: E external/xla/xla/python/profiler/internal/python_hooks.cc:412] Can't import tensorflow.python.profiler.trace
2026-07-16 12:15:11.239399: E external/xla/xla/python/profiler/internal/python_hooks.cc:412] Can't import tensorflow.python.profiler.trace
2026-07-16 12:15:13.123311: E external/xla/xla/python/profiler/internal/python_hooks.cc:412] Can't import tensorflow.python.profiler.trace


In [55]:
%tensorboard --logdir /tmp/tensorboard

Reusing TensorBoard on port 6006 (pid 9909), started 0:15:02 ago. (Use '!kill 9909' to kill it.)

- Raw: 388.466us
- Scan: 342.945us
- Loop: 342.570us

Some of the overlapping is happening in large matrices!

3. Put these two together into an end-to-end Transformer block that performs `In[B_X, D_Y]` $*_D$ `Win[D, F_Y]` $*_F$ `Wout[F_Y, D]` $\rightarrow$ `Out[B_X, D_Y]` with overlapped communication. How much faster is this than a `jax.jit` implementation?

In [84]:
mesh = jax.make_mesh(axis_shapes=(4, 2), axis_names=("x", "y"))
shd.set_mesh(mesh)

Mesh(device_ids=array([[0, 1, 2, 3],
       [7, 6, 5, 4]]), axis_names=('x', 'y'), axis_types=(Auto, Auto))

In [85]:
def naive_matmul_allgather(lhs, rhs):
    out = lhs @ rhs
    out = jax.lax.with_sharding_constraint(out, P("x", "y"))
    return out


def collective_matmul_allgather(lhs, rhs):
    Bx, chunk_size = lhs.shape
    _, Fy = rhs.shape

    axis_size = jax.lax.axis_size("y")
    idx = jax.lax.axis_index("y")

    assert Fy % chunk_size == 0

    def f(i, carrys):
        accum, lhs = carrys
        rhs_chunk = jax.lax.dynamic_slice_in_dim(
            rhs, (idx + i) % axis_size * chunk_size, chunk_size
        )
        update = lhs @ rhs_chunk
        lhs = jax.lax.ppermute(
            lhs,
            axis_name="y",
            perm=[(j, (j - 1) % axis_size) for j in range(axis_size)],
        )
        return accum + update, lhs

    accum = jnp.zeros((Bx, Fy), dtype=lhs.dtype)
    accum = jax.lax.pvary(accum, ("x", "y"))
    accum, lhs = jax.lax.fori_loop(0, axis_size - 1, f, (accum, lhs), unroll=True)

    i = axis_size - 1
    rhs_chunk = jax.lax.dynamic_slice_in_dim(rhs, (idx + i) % axis_size * chunk_size, chunk_size)
    update = lhs @ rhs_chunk
    return accum + update

In [86]:
jitted_naive_matmul_allgather = jax.jit(naive_matmul_allgather)
jitted_collective_matmul_allgather = jax.jit(
    jax.shard_map(
        collective_matmul_allgather,
        in_specs=(P("x", "y"), P(None, "y")),
        out_specs=P("x", "y"),
    )
)

key = jax.random.PRNGKey(0)
A = jax.random.randint(key, (1024, 2048), 0, 16).astype(jnp.bfloat16)
W1 = jax.random.randint(key, (2048, 8192), 0, 16).astype(jnp.bfloat16)

A = jax.device_put(A, P("x", "y"))
W1 = jax.device_put(W1, P(None, "y"))

with jax.profiler.trace("/tmp/tensorboard"):
    y = jitted_naive_matmul_allgather(A, W1)
    y.block_until_ready()

with jax.profiler.trace("/tmp/tensorboard"):
    y1 = jitted_collective_matmul_allgather(A, W1)
    y1.block_until_ready()

np.testing.assert_allclose(y, y1, rtol=1e-2, atol=1e-2)

2026-07-16 12:36:41.029244: E external/xla/xla/python/profiler/internal/python_hooks.cc:412] Can't import tensorflow.python.profiler.trace
2026-07-16 12:36:41.633796: E external/xla/xla/python/profiler/internal/python_hooks.cc:412] Can't import tensorflow.python.profiler.trace
2026-07-16 12:36:42.549584: E external/xla/xla/python/profiler/internal/python_hooks.cc:412] Can't import tensorflow.python.profiler.trace
2026-07-16 12:36:43.996462: E external/xla/xla/python/profiler/internal/python_hooks.cc:412] Can't import tensorflow.python.profiler.trace


In [87]:
%tensorboard --logdir /tmp/tensorboard

Reusing TensorBoard on port 6006 (pid 9909), started 0:36:33 ago. (Use '!kill 9909' to kill it.)

In [88]:
def matmul_naive(lhs, W1, W2):
    temp = lhs @ W1
    out = temp @ W2
    return out


def collective_matmul(lhs, W1, W2):
    temp = collective_matmul_allgather(lhs, W1)
    return collective_matmul_reducescatter_scan(temp, W2)

In [91]:
jitted_matmul_naive = jax.jit(
    jax.shard_map(
        matmul_naive,
        in_specs=(P("x", "y"), P(None, "y"), P("y", None)),
        out_specs=P("x", "y"),
    )
)
jitted_collective_matmul = jax.jit(
    jax.shard_map(
        collective_matmul,
        in_specs=(P("x", "y"), P(None, "y"), P("y", None)),
        out_specs=P("x", "y"),
    )
)

key = jax.random.PRNGKey(0)
A = jax.random.randint(key, (32768, 2048), 0, 16).astype(jnp.bfloat16)
W1 = jax.random.randint(key, (2048, 8192), 0, 16).astype(jnp.bfloat16)
W2 = jax.random.randint(key, (8192, 2048), 0, 16).astype(jnp.bfloat16)

A = jax.device_put(A, P("x", "y"))
W1 = jax.device_put(W1, P(None, "y"))
W2 = jax.device_put(W2, P("y", None))

with jax.profiler.trace("/tmp/tensorboard"):
    y = jitted_collective_matmul(A, W1, W2)
    y.block_until_ready()

with jax.profiler.trace("/tmp/tensorboard"):
    y1 = matmul_naive(A, W1, W2)
    y1.block_until_ready()

np.testing.assert_allclose(y, y1, rtol=2e-2, atol=1e-2)

2026-07-16 12:38:41.602287: E external/xla/xla/python/profiler/internal/python_hooks.cc:412] Can't import tensorflow.python.profiler.trace
2026-07-16 12:38:45.056633: E external/xla/xla/python/profiler/internal/python_hooks.cc:412] Can't import tensorflow.python.profiler.trace
2026-07-16 12:38:46.107915: E external/xla/xla/python/profiler/internal/python_hooks.cc:412] Can't import tensorflow.python.profiler.trace
2026-07-16 12:38:46.201158: E external/xla/xla/python/profiler/internal/python_hooks.cc:412] Can't import tensorflow.python.profiler.trace


In [92]:
%tensorboard --logdir /tmp/tensorboard

Reusing TensorBoard on port 6006 (pid 9909), started 0:38:35 ago. (Use '!kill 9909' to kill it.)

Raw: 974.595us
Collective: 656.905us

48.36% faster!

faster in (4, 2) mesh, slower in (2, 4) mesh

## Question 4

All of the collective matmuls implemented above are unidirectional: they only permute in one direction. Rewrite the collective AllReduce matmul and the collective ReduceScatter matmuls to use bidirectional communication. How much faster are these?

In [120]:
mesh = jax.make_mesh(axis_shapes=(2, 4), axis_names=("x", "y"))
shd.set_mesh(mesh)

Mesh(device_ids=array([[0, 1],
       [2, 3],
       [7, 6],
       [5, 4]]), axis_names=('x', 'y'), axis_types=(Auto, Auto))

In [121]:
def collective_matmul_reducescatter_bidirectional(
    lhs: jnp.ndarray, rhs: jnp.ndarray
) -> jnp.ndarray:
    B, _ = lhs.shape
    _, D = rhs.shape

    axis_size = jax.lax.axis_size("y")
    idx = jax.lax.axis_index("y")

    chunk_size = D // axis_size

    up_perm = [(j, (j - 1) % axis_size) for j in range(axis_size)]
    down_perm = [(j, (j + 1) % axis_size) for j in range(axis_size)]

    lhs_up = jax.lax.dynamic_slice_in_dim(lhs, 0, B // 2, axis=0)
    lhs_down = jax.lax.dynamic_slice_in_dim(lhs, B // 2, B // 2, axis=0)

    def body_fn(carry: tuple[jnp.ndarray, jnp.ndarray], i: int):
        accum_up, accum_down = carry
        accum_up = jax.lax.ppermute(accum_up, axis_name="y", perm=up_perm)
        accum_down = jax.lax.ppermute(accum_down, axis_name="y", perm=down_perm)

        rhs_chunk_up = jax.lax.dynamic_slice_in_dim(
            rhs, (idx + i + 1) % axis_size * chunk_size, chunk_size, axis=1
        )
        rhs_chunk_down = jax.lax.dynamic_slice_in_dim(
            rhs, (idx - i - 1) % axis_size * chunk_size, chunk_size, axis=1
        )
        update_up = lhs_up @ rhs_chunk_up
        update_down = lhs_down @ rhs_chunk_down

        return (accum_up + update_up, accum_down + update_down), (update_up, update_down)

    rhs_chunk_up = jax.lax.dynamic_slice_in_dim(
        rhs, (idx + 1) % axis_size * chunk_size, chunk_size, axis=1
    )
    rhs_chunk_down = jax.lax.dynamic_slice_in_dim(
        rhs, (idx - 1) % axis_size * chunk_size, chunk_size, axis=1
    )
    accum_up = lhs_up @ rhs_chunk_up
    accum_down = lhs_down @ rhs_chunk_down
    (out_up, out_down), _ = jax.lax.scan(
        body_fn, (accum_up, accum_down), jnp.arange(1, axis_size), unroll=True
    )

    return jnp.concatenate([out_up, out_down], axis=0)

In [122]:
jitted_collective_matmul_reducescatter = jax.jit(
    jax.shard_map(
        collective_matmul_reducescatter_scan,
        in_specs=(P("x", "y"), P("y", None)),
        out_specs=P("x", "y"),
    )
)
jitted_collective_matmul_reducescatter_bidirectional = jax.jit(
    jax.shard_map(
        collective_matmul_reducescatter_bidirectional,
        in_specs=(P("x", "y"), P("y", None)),
        out_specs=P("x", "y"),
    )
)

A = jax.random.randint(key, (16384, 2048), 0, 16).astype(jnp.bfloat16)
B = jax.random.randint(key, (2048, 8192), 0, 16).astype(jnp.bfloat16)

A = jax.device_put(A, P("x", "y"))
B = jax.device_put(B, P("y", None))

with jax.profiler.trace("/tmp/tensorboard"):
    y = jitted_collective_matmul_reducescatter(A, B)
    y.block_until_ready()

with jax.profiler.trace("/tmp/tensorboard"):
    y1 = jitted_collective_matmul_reducescatter_bidirectional(A, B)
    y1.block_until_ready()

np.testing.assert_allclose(y, y1, rtol=1e-2, atol=1e-2)

2026-07-16 12:57:19.358349: E external/xla/xla/python/profiler/internal/python_hooks.cc:412] Can't import tensorflow.python.profiler.trace
2026-07-16 12:57:20.915242: E external/xla/xla/python/profiler/internal/python_hooks.cc:412] Can't import tensorflow.python.profiler.trace
2026-07-16 12:57:21.902757: E external/xla/xla/python/profiler/internal/python_hooks.cc:412] Can't import tensorflow.python.profiler.trace
2026-07-16 12:57:23.175835: E external/xla/xla/python/profiler/internal/python_hooks.cc:412] Can't import tensorflow.python.profiler.trace


In [123]:
%tensorboard --logdir /tmp/tensorboard

Reusing TensorBoard on port 6006 (pid 9909), started 0:57:13 ago. (Use '!kill 9909' to kill it.)

Wrote the bidirectional version for the ReduceScatter version since AllReduce only uses `psum` not `ppermute`. 

(4, 2) mesh
- Unidirectional: 508.031us
- Bidirectional: 460.269us

About 10% faster

(2, 4) mesh
- Unidirectional: 1.243ms
- Bidirectional: 1.199ms

About 3.7% faster


faster since we utilize bidirectional bandwidth! similar in small matrices